# 04. Training-only augmentation

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.transfer import transfer
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
cohorts = list(transfer.FULL_COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)

In [ ]:
features_run = ""  # Optional pipeline_outputs/transfer/features_<id>; blank uses the latest
features_run = Path(features_run) if features_run else tuning.latest_run(root, "features", area=transfer.AREA)
feature_choice = json.loads((features_run / "selected.json").read_text())
print("features run:", features_run)
feature_choice

## Augmentation settings

In [ ]:
augmentations = {
    "none": None,
    "shift_1_bin": {"shifts": (-1, 1)},
    "band_gain_10pct": {"band_spread": .10, "copies": 2},
    "shift_and_band_gain": {"shifts": (-1, 1), "band_spread": .10, "copies": 2},
}
augmentations

## Spatial cross-validation, locked validation and leave-one-source-out

In [ ]:
run = tuning.new_run(root, "augmentation", features_path, area=transfer.AREA,
                     features_run=str(features_run), augmentations={k: str(v) for k, v in augmentations.items()})
records, holdout = [], []
for cohort in cohorts:
    variant = feature_choice[cohort]["series_variant"]
    params = {**feature_choice[cohort]["params"], "n_estimators": 300}
    train, validation = tuning.cohort_split(development, cohort)
    rows = train + validation
    for name, augmentation in augmentations.items():
        pooled = transfer.spatial_cv(train, params, variant, augmentation)
        locked = transfer.locked_validation(train, validation, params, variant, augmentation)
        records.append({"cohort": cohort, "variant": variant, "augmentation": name,
                        "spatial_macro_f1": pooled["macro_f1"], "fold_min": min(pooled["fold_macro_f1"]),
                        "fold_max": max(pooled["fold_macro_f1"]), "locked_validation_macro_f1": locked["macro_f1"],
                        **{f"spatial_{crop}_f1": v["f1"] for crop, v in pooled["per_crop"].items()}})
        holdout.extend({"cohort": cohort, "augmentation": name, **r}
                       for r in transfer.leave_one_source_out(rows, params, variant, augmentation))
        print(cohort, name, f"spatial {pooled['macro_f1']:.3f}", flush=True)
tuning.save_records(run, "augmentation_scores", records)
tuning.save_records(run, "augmentation_leave_one_source_out", holdout)

## Results table

In [ ]:
holdout_mean = (pd.DataFrame([r for r in holdout if r["status"] == "scored"])
                .groupby(["cohort", "augmentation"]).macro_f1_scored_crops.mean().rename("source_holdout_macro_f1_mean"))
summary = pd.DataFrame(records).set_index(["cohort", "augmentation"]).join(holdout_mean).reset_index()
summary.to_csv(run / "augmentation_summary.csv", index=False)
display(summary.round(3))
display(pd.DataFrame([r for r in holdout if r["status"] == "scored"]).round(3))

## Selected augmentation

In [ ]:
selected = {}
for cohort in cohorts:
    best = summary[summary.cohort == cohort].sort_values("spatial_macro_f1", ascending=False).iloc[0]
    selected[cohort] = {**feature_choice[cohort], "augmentation": best.augmentation,
                        "augmentation_settings": augmentations[best.augmentation]}
(run / "selected.json").write_text(json.dumps(selected, indent=2))
selected

## Figure

In [ ]:
figure, axes = plt.subplots(1, len(cohorts), figsize=(7 * len(cohorts), 4), sharey=True)
for axis, cohort in zip(axes, cohorts):
    summary[summary.cohort == cohort].set_index("augmentation")[
        ["spatial_macro_f1", "locked_validation_macro_f1", "source_holdout_macro_f1_mean"]].plot(kind="bar", ax=axis)
    axis.set_title(cohort)
    axis.set_ylim(0, 1)
show_and_save(figure, run, "augmentation")
print("Run folder:", run)